In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

# ── 호칭 추출 ─────────────────────────────────
def extract_title(df):
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don','Dr',
         'Major','Rev','Sir','Jonkheer','Dona'], 'Other'
    )
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    return df

train = extract_title(train)
test  = extract_title(test)

# ── 나이 결측치: 호칭별 중앙값으로 채우기 ─────
def fill_age(df):
    age_medians = df.groupby('Title')['Age'].median()
    df['Age'] = df.apply(
        lambda row: age_medians[row['Title']] if pd.isnull(row['Age']) else row['Age'],
        axis=1
    )
    return df

train = fill_age(train)
test  = fill_age(test)

# ── 기타 결측치 처리 ──────────────────────────
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

# ── 파생 변수 생성 ────────────────────────────
for df in [train, test]:
    df['FamilySize']  = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']     = (df['FamilySize'] == 1).astype(int)
    df['HasCabin']    = df['Cabin'].notnull().astype(int)

for df in [train, test]:
    # 1. 여성 + 등급 조합
    df['Woman_1class'] = ((df['Sex'] == 'female') & (df['Pclass'] == 1)).astype(int)
    df['Woman_2class'] = ((df['Sex'] == 'female') & (df['Pclass'] == 2)).astype(int)

    # 2. 어린이 여부
    df['IsChild'] = (df['Age'] <= 12).astype(int)

    # 3. 혼자인지 여부
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # 4. Fare 이상치 제거
    df['Fare'] = df['Fare'].clip(upper=300)

# ── 인코딩
for col in ['Sex', 'Embarked', 'Title']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# ── 최종 피처 선택 ────────────────────────────
FEATURES = ['Pclass','Sex','Age','Fare','Embarked','Title',
            'FamilySize','IsAlone','HasCabin',
            'Woman_1class','Woman_2class','IsChild']


X_train = train[FEATURES]
y_train = train['Survived']
X_test  = test[FEATURES]

print("X_train shape:", X_train.shape)
print("결측치:", X_train.isnull().sum().sum())

# 저장
X_train.to_csv('../data/X_train.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)

X_train shape: (891, 12)
결측치: 0
